# SMM4H-HeaRD 2026 – Task 2, Subtask 2 — Improved v2
## Multi-label Classification + Evidence Span Extraction using Qwen3-8B

### What changed from v1 (based on CodaBench feedback):

| Issue | Root Cause | Fix |
|---|---|---|
| Span Exact Match ≈ 0 | Model returned long sentences; gold wants SHORT phrases (avg 8–17 chars) | Ask model for shortest verbatim phrase only |
| Rule B/C spans wrong offsets | Model returned prose text, not exact drug name | **Deterministic keyword search** for Rule B/C — no LLM needed |
| Definition 2 recall = 0.33 | Indirect symptoms need richer keyword list + LLM | Better prompting + keywords |
| Multi-occurrence missing | Only first occurrence returned | Find **ALL occurrences** of each evidence phrase |

### Architecture:
- **Rule B & Rule C**: Pure regex/keyword search (drug names are deterministic). LLM only decides the label, keywords find the spans.
- **Definition 1 & 2**: LLM extracts the **shortest verbatim phrase** from the note that constitutes the evidence. Then find ALL occurrences in the text.

In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf
print('Done.')

Done.


In [2]:
import json, re, os, time, gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#from google.colab import files

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

CUDA: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [ ]:
print('Upload: training_corpus.csv, validation_corpus.csv, test_corpus.csv,')
print('        subtask_2_train.json, subtask_2_valid.json')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [3]:
train_corpus = pd.read_csv('training_corpus.csv').set_index('note_id')
valid_corpus = pd.read_csv('validation_corpus.csv').set_index('note_id')
test_corpus  = pd.read_csv('test_corpus.csv').set_index('note_id')

with open('subtask_2_train.json') as f:
    train_labels = json.load(f)
with open('subtask_2_valid.json') as f:
    valid_labels = json.load(f)

COMPONENTS = ['Definition 1', 'Definition 2', 'Rule B', 'Rule C']

print(f'Train: {len(train_corpus)} notes | Valid: {len(valid_corpus)} | Test: {len(test_corpus)}')

# Label distribution
print('\nTraining label distribution:')
for comp in COMPONENTS:
    yes = sum(1 for v in train_labels.values() if v[comp]['label'] == 'yes')
    pct = yes / len(train_labels)
    print(f'  {comp}: yes={yes} ({pct:.0%}), no={len(train_labels)-yes}')

Train: 156 notes | Valid: 23 | Test: 1959

Training label distribution:
  Definition 1: yes=73 (47%), no=83
  Definition 2: yes=50 (32%), no=106
  Rule B: yes=37 (24%), no=119
  Rule C: yes=67 (43%), no=89


## Drug Keyword Lists
These are used for **deterministic span extraction** for Rule B and Rule C.
The model only decides yes/no — drug name locations are found by regex.

Critically, ALL occurrences of each drug name in the note are returned as spans (matching annotation style).

In [4]:
# ─── Rule B: Primary / dedicated sleep medications ────────────────────────────
# Source: annotation guidelines + training data analysis
RULE_B_DRUGS = [
    # Zolpidem (most common in MIMIC)
    'Zolpidem Tartrate', 'Zolpidem',
    'Ambien CR', 'Ambien', 'AMBIEN', 'ambien',
    'Edluar', 'Intermezzo', 'Zolpimist',
    # Triazolam
    'Triazolam', 'Halcion', 'HALCION', 'halcion',
    # Temazepam
    'Temazepam', 'Restoril',
    # Eszopiclone
    'Eszopiclone', 'Lunesta',
    # Zaleplon
    'Zaleplon', 'Sonata',
    # Flurazepam
    'Flurazepam', 'Dalmane',
    # Estazolam
    'Estazolam', 'ProSom',
    # Quazepam
    'Quazepam', 'Doral',
    # Ramelteon
    'Ramelteon', 'Rozerem',
    # Suvorexant
    'Suvorexant', 'Belsomra',
    # Lemborexant
    'Lemborexant', 'Dayvigo',
    # Doxylamine (OTC sleep aid)
    'Doxylamine', 'Unisom',
    # Oxazepam (Serax - appears in training data for sleep)
    'Oxazepam', 'Serax', 'SERAX', 'serax',
    # Chloral hydrate
    'Chloral hydrate', 'Chloral Hydrate',
]

# ─── Rule C: Secondary / off-label sleep medications ─────────────────────────
RULE_C_DRUGS = [
    # Benzodiazepines
    'Lorazepam', 'LORAZEPAM', 'Ativan', 'ATIVAN', 'ativan',
    'Diazepam', 'DIAZEPAM', 'Valium',
    'Clonazepam', 'Klonopin',
    'Alprazolam', 'Xanax',
    'Midazolam', 'Versed',
    'Clorazepate', 'Tranxene',
    'Chlordiazepoxide', 'Librium',
    # Atypical antipsychotics
    'Quetiapine', 'QUETIAPINE', 'Seroquel', 'SEROQUEL', 'SERAQUIL', 'seraquil',
    'Olanzapine', 'Zyprexa',
    # Antidepressants used for sleep
    'Trazodone', 'trazodone', 'traZODONE', 'TRAZodone', 'TRAZODONE',
    'Mirtazapine', 'Remeron',
    'Doxepin', 'Silenor',
    'Amitriptyline',
    # Antihistamines
    'Diphenhydramine', 'DIPHENHYDRAMINE', 'DiphenhydrAMINE', 'Benadryl',
    'Hydroxyzine', 'Vistaril', 'Atarax',
    # Other
    'Melatonin',
    'Gabapentin',  # sometimes used off-label for sleep
    'Pregabalin', 'Lyrica',
    'Promethazine', 'Phenergan',
]

print(f'Rule B drug list: {len(RULE_B_DRUGS)} entries')
print(f'Rule C drug list: {len(RULE_C_DRUGS)} entries')

Rule B drug list: 39 entries
Rule C drug list: 49 entries


In [5]:
# ─── Span extraction utilities ────────────────────────────────────────────────

def find_all_occurrences(text, keyword):
    """
    Find ALL occurrences of keyword in text (case-insensitive).
    Returns list of 'start end' strings matching the EXACT case in text.
    Uses word-boundary matching to avoid partial matches (e.g. 'Ativan' not inside 'Motivation').
    """
    spans = []
    # Word-boundary regex for drug names
    pattern = r'(?<![\w])' + re.escape(keyword) + r'(?![\w])'
    for m in re.finditer(pattern, text, re.IGNORECASE):
        spans.append(f"{m.start()} {m.end()}")
    return spans


def find_all_keyword_spans(text, keyword_list):
    """
    Find all occurrences of any keyword from the list in the text.
    Returns (found: bool, spans: list of 'start end', matched_drugs: list).
    Deduplicates overlapping spans.
    """
    all_spans = []  # (start, end, text)
    matched_drugs = []
    
    for kw in keyword_list:
        pattern = r'(?<![\w])' + re.escape(kw) + r'(?![\w])'
        for m in re.finditer(pattern, text, re.IGNORECASE):
            all_spans.append((m.start(), m.end(), text[m.start():m.end()]))
            if kw not in matched_drugs:
                matched_drugs.append(kw)
    
    # Sort by start position
    all_spans.sort(key=lambda x: x[0])
    
    # Remove overlapping spans (keep longer one)
    deduped = []
    prev_end = -1
    for start, end, matched_text in all_spans:
        if start >= prev_end:
            deduped.append(f"{start} {end}")
            prev_end = end
    
    return len(deduped) > 0, deduped, matched_drugs


def find_phrase_spans(text, phrase):
    """
    Find all occurrences of a phrase in text (case-insensitive).
    For Definition 1/2 evidence phrases.
    """
    spans = []
    if not phrase or len(phrase.strip()) < 3:
        return spans
    phrase = phrase.strip()
    lower_text = text.lower()
    lower_phrase = phrase.lower()
    start = 0
    while True:
        idx = lower_text.find(lower_phrase, start)
        if idx == -1:
            break
        spans.append(f"{idx} {idx + len(phrase)}")
        start = idx + 1
    return spans


print('Span extraction utilities ready.')

Span extraction utilities ready.


In [6]:
# ─── Validate keyword lists against training data ─────────────────────────────
# Check recall of our keyword lists on Rule B and Rule C training annotations

def eval_keyword_recall(drug_list, comp, labels, corpus):
    tp = fn = 0
    misses = []
    for note_id, note in labels.items():
        if note[comp]['label'] != 'yes':
            continue
        nid_int = int(note_id)
        if nid_int not in corpus.index:
            continue
        text = corpus.loc[nid_int, 'text']
        found, spans, _ = find_all_keyword_spans(text, drug_list)
        if found:
            tp += 1
        else:
            fn += 1
            misses.append((note_id, note[comp].get('text', [])))
    total = tp + fn
    print(f'{comp}: Keyword recall = {tp}/{total} = {tp/total:.2%}')
    if misses:
        print(f'  Missed gold texts: {misses[:5]}')
    return tp / total if total > 0 else 0

print('=== Training set keyword recall validation ===')
eval_keyword_recall(RULE_B_DRUGS, 'Rule B', train_labels, train_corpus)
eval_keyword_recall(RULE_C_DRUGS, 'Rule C', train_labels, train_corpus)

print('\n=== Validation set keyword recall validation ===')
eval_keyword_recall(RULE_B_DRUGS, 'Rule B', valid_labels, valid_corpus)
eval_keyword_recall(RULE_C_DRUGS, 'Rule C', valid_labels, valid_corpus)

=== Training set keyword recall validation ===


Rule B: Keyword recall = 37/37 = 100.00%
Rule C: Keyword recall = 66/67 = 98.51%
  Missed gold texts: [('1260815', ['Zyprexia', 'zyprexia'])]

=== Validation set keyword recall validation ===
Rule B: Keyword recall = 5/5 = 100.00%
Rule C: Keyword recall = 6/6 = 100.00%


1.0

In [7]:
# ─── Load Qwen3-8B-Instruct with 4-bit quantization ───────────────────────────
MODEL_NAME = 'Qwen/Qwen3-8B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print('Loading model (4-bit)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading tokenizer...


Loading model (4-bit)...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model loaded. VRAM used: 6.09 GB


In [8]:
# ─── Build few-shot examples from training data ───────────────────────────────
# Select 2 good examples: one with all yes, one with all no
# This teaches the model to return SHORT verbatim phrases

def build_few_shot_examples(train_labels, train_corpus, n_yes=1, n_no=1):
    """Select representative examples for the few-shot prompt."""
    examples = []
    
    # Find a note with both Def1=yes and Def2=yes (shows both definitions)
    for note_id, note in train_labels.items():
        if (note['Definition 1']['label'] == 'yes' and 
            note['Definition 2']['label'] == 'yes' and
            note['Definition 1'].get('text')):
            nid_int = int(note_id)
            if nid_int in train_corpus.index:
                text = train_corpus.loc[nid_int, 'text']
                # Only use first 2000 chars as example snippet
                snippet = text[:2000]
                example = {
                    'text': snippet,
                    'answer': {
                        'Definition 1': {
                            'label': 'yes',
                            'phrases': note['Definition 1'].get('text', [])[:2]
                        },
                        'Definition 2': {
                            'label': 'yes',
                            'phrases': note['Definition 2'].get('text', [])[:2]
                        }
                    }
                }
                examples.append(example)
                break
    
    # Find a note with all no
    for note_id, note in train_labels.items():
        if all(note[c]['label'] == 'no' for c in COMPONENTS):
            nid_int = int(note_id)
            if nid_int in train_corpus.index:
                text = train_corpus.loc[nid_int, 'text']
                examples.append({'text': text[:1000], 'answer': 'all_no'})
                break
    
    return examples

few_shot = build_few_shot_examples(train_labels, train_corpus)
print(f'Built {len(few_shot)} few-shot examples.')
if few_shot and few_shot[0].get('answer') != 'all_no':
    print('Example 1 Def1 phrases:', few_shot[0]['answer']['Definition 1']['phrases'])
    print('Example 1 Def2 phrases:', few_shot[0]['answer']['Definition 2']['phrases'])

Built 2 few-shot examples.
Example 1 Def1 phrases: ['Insomnia', 'insomnia']
Example 1 Def2 phrases: ['Insomnia', 'insomnia']


In [9]:
# ─── Prompt template (improved for short phrase extraction) ───────────────────

SYSTEM_PROMPT = """You are a clinical NLP expert. Analyze clinical notes to identify insomnia evidence.

RULES TO EVALUATE:
- Definition 1 (DIRECT insomnia): Patient explicitly cannot sleep. Keywords: insomnia, unable to sleep, can't sleep, difficulty sleeping, sleeplessness, not sleeping, sleep disturbance, difficulty falling asleep, difficulty staying asleep, poor sleep, trouble sleeping, awake all night, no sleep, waking frequently, c/o inability to sleep.
- Definition 2 (INDIRECT/DAYTIME effects): Patient has secondary sleep-related complaint: fatigue, exhaustion, restlessness, agitation, anxiety about sleep, depression preventing sleep, daytime sleepiness, lethargic, tired due to lack of sleep.
- Rule B (PRIMARY sleep drug prescribed): Zolpidem/Ambien, Triazolam/Halcion, Temazepam/Restoril, Eszopiclone/Lunesta, Zaleplon/Sonata, Flurazepam/Dalmane, Oxazepam/Serax, Ramelteon/Rozerem.
- Rule C (SECONDARY/off-label sleep drug prescribed): Lorazepam/Ativan, Diazepam/Valium, Clonazepam/Klonopin, Midazolam/Versed, Quetiapine/Seroquel, Trazodone, Mirtazapine/Remeron, Diphenhydramine/Benadryl, Hydroxyzine/Vistaril, Doxepin.

OUTPUT RULES (CRITICAL):
1. Return ONLY valid JSON, no other text.
2. For 'phrases': provide the SHORTEST verbatim text from the note that proves the rule. Use 1-5 words typically.
3. For drug names (Rule B/C): the phrase is JUST the drug name as it appears (e.g. 'Zolpidem', 'AMBIEN', 'Lorazepam').
4. Do NOT paraphrase. Copy exact text from the note.
5. If label is 'no', phrases must be empty list [].
6. Only label 'yes' if evidence is CLEARLY present.

JSON FORMAT:
{"Definition 1": {"label": "yes"|"no", "phrases": ["exact phrase from note"]},
 "Definition 2": {"label": "yes"|"no", "phrases": ["exact phrase from note"]},
 "Rule B":       {"label": "yes"|"no", "phrases": ["drug name as in note"]},
 "Rule C":       {"label": "yes"|"no", "phrases": ["drug name as in note"]}}"""


def build_user_prompt(text, max_chars=11000):
    # Keep first 11000 chars (most relevant info is at start and discharge section)
    if len(text) > max_chars:
        # Keep beginning and end (discharge section often has key info)
        half = max_chars // 2
        text = text[:half] + '\n\n[...middle truncated...]\n\n' + text[-half:]
    return f"""Clinical note to analyze:\n\n{text}\n\nReturn JSON only:"""


def run_inference(text, max_new_tokens=512, temperature=0.05):
    """Run model inference. Low temperature for more deterministic structured output."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_prompt(text)}
    ]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # Qwen3: disable chain-of-thought for structured output
    )
    inputs = tokenizer(
        formatted, return_tensors='pt', truncation=True, max_length=5500
    ).to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


print('Prompt and inference functions ready.')

Prompt and inference functions ready.


In [10]:
# ─── Output parsing and span assembly ────────────────────────────────────────

def parse_model_output(raw_output):
    """Parse JSON from model output. Returns dict or None."""
    # Strip thinking block if present (Qwen3)
    raw = re.sub(r'<think>[\s\S]*?</think>', '', raw_output).strip()
    
    # Extract JSON object
    json_match = re.search(r'\{[\s\S]*\}', raw)
    if not json_match:
        return None
    
    json_str = json_match.group(0)
    
    # Fix common JSON issues
    json_str = re.sub(r',\s*}', '}', json_str)   # trailing commas
    json_str = re.sub(r',\s*]', ']', json_str)   # trailing commas in arrays
    
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            # Last resort: eval (safe since we control input)
            return json.loads(json_str.replace("'", '"'))
        except:
            return None


def assemble_result(note_text, model_output, use_drug_keywords=True):
    """
    Core function: combine model predictions with deterministic span extraction.
    
    Strategy per component:
    - Rule B/C: Model decides label. Spans come from keyword search (ALL occurrences).
    - Def 1/2:  Model decides label AND provides short phrase. Span = find phrase in text.
    """
    # Fallback if model output is unusable
    fallback = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}
    
    if model_output is None:
        # Fall back to pure keyword-based for Rule B/C at minimum
        result = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}
        if use_drug_keywords:
            for comp, drug_list in [('Rule B', RULE_B_DRUGS), ('Rule C', RULE_C_DRUGS)]:
                found, spans, _ = find_all_keyword_spans(note_text, drug_list)
                if found:
                    result[comp] = {'label': 'yes', 'span': spans}
        return result
    
    result = {}
    
    # ── Definition 1 & Definition 2: LLM label + LLM phrase → span search ─────
    for comp in ['Definition 1', 'Definition 2']:
        comp_data = model_output.get(comp, {})
        label = str(comp_data.get('label', 'no')).lower().strip()
        if label not in ('yes', 'no'):
            label = 'no'
        
        if label == 'no':
            result[comp] = {'label': 'no', 'span': []}
            continue
        
        # Find spans for each evidence phrase
        phrases = comp_data.get('phrases', [])
        all_spans = []
        for phrase in phrases:
            if isinstance(phrase, str) and len(phrase.strip()) >= 3:
                # Try exact phrase
                spans = find_phrase_spans(note_text, phrase)
                if not spans and len(phrase.split()) > 3:
                    # Try a shorter core (first 3 meaningful words)
                    core_words = phrase.strip().split()[:3]
                    core = ' '.join(core_words)
                    spans = find_phrase_spans(note_text, core)
                all_spans.extend(spans)
        
        # Deduplicate
        all_spans = list(dict.fromkeys(all_spans))
        
        if not all_spans:
            # Demote to no - can't find span, label constraint violated
            result[comp] = {'label': 'no', 'span': []}
        else:
            result[comp] = {'label': 'yes', 'span': all_spans}
    
    # ── Rule B: LLM label + keyword search for ALL drug name occurrences ───────
    rule_b_data = model_output.get('Rule B', {})
    rule_b_label = str(rule_b_data.get('label', 'no')).lower().strip()
    
    if use_drug_keywords:
        # Always run keyword search regardless of model label
        found_b, spans_b, _ = find_all_keyword_spans(note_text, RULE_B_DRUGS)
        
        # Model says yes OR keyword found → yes (high recall)
        # Model says no AND no keyword found → no
        final_b = rule_b_label == 'yes' or found_b
        
        if final_b and spans_b:
            result['Rule B'] = {'label': 'yes', 'span': spans_b}
        elif rule_b_label == 'yes' and not spans_b:
            # Model says yes but no keyword found - check phrases from model
            phrases = rule_b_data.get('phrases', [])
            fallback_spans = []
            for ph in phrases:
                if isinstance(ph, str):
                    s = find_phrase_spans(note_text, ph.strip())
                    fallback_spans.extend(s)
            if fallback_spans:
                result['Rule B'] = {'label': 'yes', 'span': list(dict.fromkeys(fallback_spans))}
            else:
                result['Rule B'] = {'label': 'no', 'span': []}
        else:
            result['Rule B'] = {'label': 'no', 'span': []}
    else:
        result['Rule B'] = {'label': rule_b_label, 'span': []}
    
    # ── Rule C: same as Rule B ─────────────────────────────────────────────────
    rule_c_data = model_output.get('Rule C', {})
    rule_c_label = str(rule_c_data.get('label', 'no')).lower().strip()
    
    if use_drug_keywords:
        found_c, spans_c, _ = find_all_keyword_spans(note_text, RULE_C_DRUGS)
        
        final_c = rule_c_label == 'yes' or found_c
        
        if final_c and spans_c:
            result['Rule C'] = {'label': 'yes', 'span': spans_c}
        elif rule_c_label == 'yes' and not spans_c:
            phrases = rule_c_data.get('phrases', [])
            fallback_spans = []
            for ph in phrases:
                if isinstance(ph, str):
                    s = find_phrase_spans(note_text, ph.strip())
                    fallback_spans.extend(s)
            if fallback_spans:
                result['Rule C'] = {'label': 'yes', 'span': list(dict.fromkeys(fallback_spans))}
            else:
                result['Rule C'] = {'label': 'no', 'span': []}
        else:
            result['Rule C'] = {'label': 'no', 'span': []}
    else:
        result['Rule C'] = {'label': rule_c_label, 'span': []}
    
    return result


print('Parsing and assembly functions ready.')

Parsing and assembly functions ready.


In [11]:
# ─── Evaluation metrics ────────────────────────────────────────────────────────

def expand_spans(spans):
    """Handle both 'start end' and 'start;end' format spans."""
    expanded = []
    for s in spans:
        for part in str(s).split(';'):
            part = part.strip()
            if re.match(r'^\d+ \d+$', part):
                expanded.append(part)
    return expanded


def spans_overlap(s1, s2):
    a, b = map(int, s1.split())
    c, d = map(int, s2.split())
    return a < d and c < b


def compute_label_metrics(predictions, gold):
    tp = fp = fn = 0
    for note_id in gold:
        for comp in COMPONENTS:
            g = gold[note_id][comp]['label']
            p = predictions.get(note_id, {}).get(comp, {}).get('label', 'no')
            if g == 'yes' and p == 'yes': tp += 1
            elif g == 'no' and p == 'yes': fp += 1
            elif g == 'yes' and p == 'no': fn += 1
    prec = tp/(tp+fp) if tp+fp > 0 else 0
    rec  = tp/(tp+fn) if tp+fn > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if prec+rec > 0 else 0
    return {'P': prec, 'R': rec, 'F1': f1, 'tp': tp, 'fp': fp, 'fn': fn}


def compute_span_metrics(predictions, gold, match='partial'):
    tp = fp = fn = 0
    for note_id in gold:
        for comp in COMPONENTS:
            g_spans = expand_spans(gold[note_id][comp].get('span', []))
            p_spans = expand_spans(predictions.get(note_id, {}).get(comp, {}).get('span', []))
            matched_g, matched_p = set(), set()
            for i, gs in enumerate(g_spans):
                for j, ps in enumerate(p_spans):
                    hit = (gs == ps) if match == 'exact' else spans_overlap(gs, ps)
                    if hit:
                        matched_g.add(i)
                        matched_p.add(j)
            tp += len(matched_g)
            fp += len(p_spans) - len(matched_p)
            fn += len(g_spans) - len(matched_g)
    prec = tp/(tp+fp) if tp+fp > 0 else 0
    rec  = tp/(tp+fn) if tp+fn > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if prec+rec > 0 else 0
    return {'P': prec, 'R': rec, 'F1': f1}


def evaluate(predictions, gold, name='Validation'):
    lm = compute_label_metrics(predictions, gold)
    se = compute_span_metrics(predictions, gold, 'exact')
    sp = compute_span_metrics(predictions, gold, 'partial')
    print(f'\n=== {name} ===')
    print(f'Label   P={lm["P"]:.4f}  R={lm["R"]:.4f}  F1={lm["F1"]:.4f}  (TP={lm["tp"]} FP={lm["fp"]} FN={lm["fn"]})')
    print(f'Span-E  P={se["P"]:.4f}  R={se["R"]:.4f}  F1={se["F1"]:.4f}')
    print(f'Span-P  P={sp["P"]:.4f}  R={sp["R"]:.4f}  F1={sp["F1"]:.4f}')
    # Per-component breakdown
    print('\n  Per-component label accuracy:')
    for comp in COMPONENTS:
        c_tp = c_fp = c_fn = 0
        for nid in gold:
            g = gold[nid][comp]['label']
            p = predictions.get(nid, {}).get(comp, {}).get('label', 'no')
            if g=='yes' and p=='yes': c_tp += 1
            elif g=='no' and p=='yes': c_fp += 1
            elif g=='yes' and p=='no': c_fn += 1
        c_p = c_tp/(c_tp+c_fp) if c_tp+c_fp>0 else 0
        c_r = c_tp/(c_tp+c_fn) if c_tp+c_fn>0 else 0
        c_f = 2*c_p*c_r/(c_p+c_r) if c_p+c_r>0 else 0
        print(f'    {comp:15s}: P={c_p:.3f} R={c_r:.3f} F1={c_f:.3f}')
    return lm, se, sp


print('Evaluation functions ready.')

Evaluation functions ready.


In [12]:
# ─── Quick Sanity Check: Keyword-only baseline on validation ──────────────────
# Rule B/C are pure keywords. Def1/Def2 are 'no' (no LLM yet).
# This shows us the floor for Rule B/C spans.

keyword_only_preds = {}
for note_id in valid_labels:
    nid_int = int(note_id)
    if nid_int not in valid_corpus.index:
        keyword_only_preds[note_id] = {comp: {'label':'no','span':[]} for comp in COMPONENTS}
        continue
    text = valid_corpus.loc[nid_int, 'text']
    
    result = {comp: {'label': 'no', 'span': []} for comp in COMPONENTS}
    found_b, spans_b, _ = find_all_keyword_spans(text, RULE_B_DRUGS)
    if found_b:
        result['Rule B'] = {'label': 'yes', 'span': spans_b}
    found_c, spans_c, _ = find_all_keyword_spans(text, RULE_C_DRUGS)
    if found_c:
        result['Rule C'] = {'label': 'yes', 'span': spans_c}
    keyword_only_preds[note_id] = result

print('Keyword-only baseline (Rule B/C only, Def1/Def2=no):')
evaluate(keyword_only_preds, valid_labels, 'Keyword-only Baseline')

Keyword-only baseline (Rule B/C only, Def1/Def2=no):

=== Keyword-only Baseline ===
Label   P=0.5500  R=0.4783  F1=0.5116  (TP=11 FP=9 FN=12)
Span-E  P=0.3800  R=0.5135  F1=0.4368
Span-P  P=0.4400  R=0.5946  F1=0.5057

  Per-component label accuracy:
    Definition 1   : P=0.000 R=0.000 F1=0.000
    Definition 2   : P=0.000 R=0.000 F1=0.000
    Rule B         : P=1.000 R=1.000 F1=1.000
    Rule C         : P=0.400 R=1.000 F1=0.571


({'P': 0.55,
  'R': 0.4782608695652174,
  'F1': 0.5116279069767442,
  'tp': 11,
  'fp': 9,
  'fn': 12},
 {'P': 0.38, 'R': 0.5135135135135135, 'F1': 0.4367816091954023},
 {'P': 0.44, 'R': 0.5945945945945946, 'F1': 0.5057471264367815})

In [13]:
# ─── Run Full Pipeline on Validation Set ──────────────────────────────────────

valid_predictions = {}
valid_ids = list(valid_labels.keys())
print(f'Running on {len(valid_ids)} validation notes...')

for i, note_id in enumerate(valid_ids):
    nid_int = int(note_id)
    if nid_int not in valid_corpus.index:
        valid_predictions[note_id] = {comp: {'label':'no','span':[]} for comp in COMPONENTS}
        print(f'  [{i+1}] {note_id}: NOT IN CORPUS')
        continue
    
    text = valid_corpus.loc[nid_int, 'text']
    
    try:
        raw = run_inference(text)
        parsed = parse_model_output(raw)
        result = assemble_result(text, parsed)
        valid_predictions[note_id] = result
        
        labels_str = ' | '.join(
            f'{c.split()[0][0]}{c.split()[1] if len(c.split())>1 else ""}:{result[c]["label"]}'
            for c in COMPONENTS
        )
        print(f'  [{i+1}/{len(valid_ids)}] {note_id}: {labels_str}')
    except Exception as e:
        print(f'  [{i+1}/{len(valid_ids)}] {note_id}: ERROR - {e}')
        # Fallback to keyword-only
        result = {comp: {'label':'no','span':[]} for comp in COMPONENTS}
        found_b, spans_b, _ = find_all_keyword_spans(text, RULE_B_DRUGS)
        if found_b: result['Rule B'] = {'label':'yes','span':spans_b}
        found_c, spans_c, _ = find_all_keyword_spans(text, RULE_C_DRUGS)
        if found_c: result['Rule C'] = {'label':'yes','span':spans_c}
        valid_predictions[note_id] = result
    
    if (i+1) % 5 == 0:
        gc.collect(); torch.cuda.empty_cache()

print('\nValidation inference complete.')

Running on 23 validation notes...


  [1/23] 10686: D1:no | D2:no | RB:no | RC:no
  [2/23] 1262007: D1:yes | D2:no | RB:yes | RC:yes
  [3/23] 1272946: D1:yes | D2:no | RB:no | RC:yes
  [4/23] 1279443: D1:yes | D2:no | RB:yes | RC:no
  [5/23] 1281746: D1:no | D2:no | RB:no | RC:no
  [6/23] 1284059: D1:yes | D2:yes | RB:no | RC:yes
  [7/23] 1288748: D1:yes | D2:no | RB:yes | RC:yes
  [8/23] 1320608: D1:yes | D2:no | RB:yes | RC:yes
  [9/23] 1324558: D1:no | D2:no | RB:no | RC:yes
  [10/23] 1328271: D1:no | D2:no | RB:yes | RC:yes
  [11/23] 1328300: D1:yes | D2:no | RB:yes | RC:yes
  [12/23] 1333759: D1:yes | D2:no | RB:no | RC:yes
  [13/23] 1334025: D1:yes | D2:yes | RB:yes | RC:yes
  [14/23] 2073: D1:no | D2:no | RB:no | RC:no
  [15/23] 24581: D1:no | D2:no | RB:no | RC:yes
  [16/23] 25898: D1:no | D2:no | RB:no | RC:yes
  [17/23] 45094: D1:no | D2:no | RB:no | RC:no
  [18/23] 54155: D1:no | D2:no | RB:no | RC:yes
  [19/23] 56587: D1:no | D2:no | RB:yes | RC:yes
  [20/23] 56803: D1:no | D2:no | RB:no | RC:no
  [21/23] 574

In [14]:
# ─── Evaluate Validation Results ──────────────────────────────────────────────
lm, se, sp = evaluate(valid_predictions, valid_labels, 'Full Pipeline Validation')

print('\n=== Detailed Error Analysis ===')
for nid in valid_labels:
    if nid not in valid_predictions:
        continue
    for comp in COMPONENTS:
        g_label = valid_labels[nid][comp]['label']
        p_label = valid_predictions[nid][comp]['label']
        if g_label != p_label:
            g_text = valid_labels[nid][comp].get('text', [])
            p_spans = valid_predictions[nid][comp].get('span', [])
            print(f'  Note {nid} | {comp}: Gold={g_label}, Pred={p_label}')
            if g_text: print(f'    Gold evidence: {g_text}')
            if p_spans: print(f'    Pred spans: {p_spans[:3]}')


=== Full Pipeline Validation ===
Label   P=0.5882  R=0.8696  F1=0.7018  (TP=20 FP=14 FN=3)
Span-E  P=0.3014  R=0.5946  F1=0.4000
Span-P  P=0.4247  R=0.8378  F1=0.5636

  Per-component label accuracy:
    Definition 1   : P=1.000 R=1.000 F1=1.000
    Definition 2   : P=0.000 R=0.000 F1=0.000
    Rule B         : P=0.625 R=1.000 F1=0.769
    Rule C         : P=0.400 R=1.000 F1=0.571

=== Detailed Error Analysis ===
  Note 1262007 | Rule C: Gold=no, Pred=yes
    Pred spans: ['439 448', '832 838']
  Note 1272946 | Definition 2: Gold=yes, Pred=no
    Gold evidence: ['upset with what has been going on with her', 'Restless']
  Note 1272946 | Rule C: Gold=no, Pred=yes
    Pred spans: ['112 121']
  Note 1279443 | Definition 2: Gold=yes, Pred=no
    Gold evidence: ['insomnia']
  Note 1284059 | Definition 2: Gold=no, Pred=yes
    Pred spans: ['817 836']
  Note 1320608 | Rule B: Gold=no, Pred=yes
    Pred spans: ['98 117', '417 429', '1663 1675']
  Note 1324558 | Rule C: Gold=no, Pred=yes
    Pre

In [15]:
# ─── Check Rule C False Positives ─────────────────────────────────────────────
# Rule C has high recall but potentially low precision due to keywords appearing
# in unrelated contexts (e.g., drug prescribed for anxiety, not sleep)
# The annotation guidelines likely require the drug to be CONNECTED to sleep context

# Check on validation: how many Rule C FP are there?
print('Rule C prediction analysis:')
for nid in valid_labels:
    g = valid_labels[nid]['Rule C']['label']
    p = valid_predictions.get(nid, {}).get('Rule C', {}).get('label', 'no')
    if g == 'no' and p == 'yes':
        preds = valid_predictions[nid]['Rule C']['span'][:3]
        # Show what text was matched
        nid_int = int(nid)
        text = valid_corpus.loc[nid_int, 'text'] if nid_int in valid_corpus.index else ''
        matched = [text[int(s.split()[0]):int(s.split()[1])] for s in preds]
        print(f'  FP {nid}: matched={matched[:3]}')

Rule C prediction analysis:
  FP 1262007: matched=['Midazolam', 'versed']
  FP 1272946: matched=['Midazolam']
  FP 1324558: matched=['traZODONE']
  FP 1328271: matched=['Diazepam', 'Lorazepam', 'Quetiapine']
  FP 24581: matched=['Lorazepam']
  FP 25898: matched=['Lorazepam', 'DiphenhydrAMINE', 'Midazolam']
  FP 54155: matched=['Zyprexa', 'Seroquel', 'Seroquel']
  FP 56587: matched=['Ativan']
  FP 58969: matched=['Trazodone', 'Amitriptyline', 'Klonopin']


In [16]:
# ─── Optional: High-Precision Mode for Rule C ─────────────────────────────────
# If Rule C precision is low, we can require LLM confirmation for the label
# (keyword finds spans, but BOTH model and keyword must agree for 'yes')
# Test both modes and pick the one with better F1

def assemble_result_strict_c(note_text, model_output):
    """Stricter mode: Rule C label requires BOTH model yes AND keyword found."""
    result = assemble_result(note_text, model_output, use_drug_keywords=True)
    
    if model_output is None:
        return result
    
    # For Rule C: require model confirmation
    rule_c_model_label = str(model_output.get('Rule C', {}).get('label', 'no')).lower()
    if result['Rule C']['label'] == 'yes' and rule_c_model_label != 'yes':
        # Keyword found but model disagrees - still keep (keyword is reliable)
        # Change this to demote if too many FPs:
        # result['Rule C'] = {'label': 'no', 'span': []}
        pass
    
    return result


# The v1 results showed Rule C precision=0.47 but recall=1.0
# This suggests keywords are finding drugs that don't indicate sleep context
# According to annotation guidelines, the drug just needs to be PRESCRIBED,
# not necessarily prescribed FOR sleep - so high recall approach may be correct.
# Keep generous mode (OR logic) for now.
print('Rule C: Using OR logic (keyword OR model) → higher recall, moderate precision')
print('To switch to AND logic (keyword AND model) → higher precision, lower recall')
print('Choice depends on target metric - F1 considers both.')

Rule C: Using OR logic (keyword OR model) → higher recall, moderate precision
To switch to AND logic (keyword AND model) → higher precision, lower recall
Choice depends on target metric - F1 considers both.


In [17]:
# ─── Run on Full Test Set ──────────────────────────────────────────────────────
# 1,959 notes. Estimated time: ~3-5 hours on T4.
# Checkpoints saved every 50 notes.

CHECKPOINT_FILE = 'test_preds_v2_checkpoint.json'
CHECKPOINT_INTERVAL = 50

# Load checkpoint if available
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        test_predictions = json.load(f)
    print(f'Checkpoint loaded: {len(test_predictions)} notes done.')
else:
    test_predictions = {}
    print('Starting fresh.')

test_ids = [str(nid) for nid in test_corpus.index.tolist()]
remaining = [nid for nid in test_ids if nid not in test_predictions]
print(f'Remaining: {len(remaining)}/{len(test_ids)}')

t0 = time.time()

for i, note_id in enumerate(remaining):
    nid_int = int(note_id)
    
    if nid_int not in test_corpus.index:
        test_predictions[note_id] = {comp: {'label':'no','span':[]} for comp in COMPONENTS}
        continue
    
    text = test_corpus.loc[nid_int, 'text']
    
    try:
        raw = run_inference(text)
        parsed = parse_model_output(raw)
        result = assemble_result(text, parsed)
        test_predictions[note_id] = result
    except Exception as e:
        # Keyword fallback on error
        result = {comp: {'label':'no','span':[]} for comp in COMPONENTS}
        found_b, spans_b, _ = find_all_keyword_spans(text, RULE_B_DRUGS)
        if found_b: result['Rule B'] = {'label':'yes','span':spans_b}
        found_c, spans_c, _ = find_all_keyword_spans(text, RULE_C_DRUGS)
        if found_c: result['Rule C'] = {'label':'yes','span':spans_c}
        test_predictions[note_id] = result
        print(f'  Error on {note_id}: {e}')
    
    # Progress
    if (i+1) % 10 == 0:
        elapsed = time.time() - t0
        eta_min = (len(remaining) - i - 1) / ((i+1)/elapsed) / 60
        print(f'  [{i+1}/{len(remaining)}] ETA: {eta_min:.0f} min')
    
    # Checkpoint
    if (i+1) % CHECKPOINT_INTERVAL == 0:
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump(test_predictions, f)
        print(f'  Checkpoint saved ({len(test_predictions)} notes).')
    
    if (i+1) % 20 == 0:
        gc.collect(); torch.cuda.empty_cache()

# Final checkpoint
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump(test_predictions, f)
print(f'\nDone! {len(test_predictions)} notes. Time: {(time.time()-t0)/60:.1f} min')

Starting fresh.
Remaining: 1959/1959
  [10/1959] ETA: 372 min
  [20/1959] ETA: 362 min
  [30/1959] ETA: 359 min
  [40/1959] ETA: 348 min
  [50/1959] ETA: 347 min
  Checkpoint saved (50 notes).
  [60/1959] ETA: 339 min
  [70/1959] ETA: 335 min
  [80/1959] ETA: 334 min
  [90/1959] ETA: 333 min
  [100/1959] ETA: 331 min
  Checkpoint saved (100 notes).
  [110/1959] ETA: 330 min
  [120/1959] ETA: 331 min
  [130/1959] ETA: 327 min
  [140/1959] ETA: 324 min
  [150/1959] ETA: 320 min
  Checkpoint saved (150 notes).
  [160/1959] ETA: 316 min
  [170/1959] ETA: 314 min
  [180/1959] ETA: 312 min
  [190/1959] ETA: 310 min
  [200/1959] ETA: 308 min
  Checkpoint saved (200 notes).
  [210/1959] ETA: 307 min
  [220/1959] ETA: 306 min
  [230/1959] ETA: 304 min
  [240/1959] ETA: 302 min
  [250/1959] ETA: 300 min
  Checkpoint saved (250 notes).
  [260/1959] ETA: 298 min
  [270/1959] ETA: 295 min
  [280/1959] ETA: 293 min
  [290/1959] ETA: 292 min
  [300/1959] ETA: 289 min
  Checkpoint saved (300 notes).
 

In [19]:
# ─── Format and Validate Submission ───────────────────────────────────────────

def build_submission(predictions, test_note_ids):
    """Build final submission dict, ensuring all test notes are present."""
    submission = {}
    for note_id in test_note_ids:
        nid_str = str(note_id)
        pred = predictions.get(nid_str, {comp: {'label':'no','span':[]} for comp in COMPONENTS})
        submission[nid_str] = {}
        for comp in COMPONENTS:
            label = pred.get(comp, {}).get('label', 'no')
            spans = pred.get(comp, {}).get('span', [])
            # Enforce constraint
            if label == 'yes' and not spans: label = 'no'
            if label == 'no': spans = []
            submission[nid_str][comp] = {'label': label, 'span': spans}
    return submission


def validate_submission(submission):
    errors = []
    for nid, note in submission.items():
        for comp in COMPONENTS:
            if comp not in note:
                errors.append(f'{nid}/{comp}: missing')
                continue
            label = note[comp].get('label')
            spans = note[comp].get('span', [])
            if label not in ('yes','no'):
                errors.append(f'{nid}/{comp}: bad label={label}')
            if label=='yes' and not spans:
                errors.append(f'{nid}/{comp}: yes but empty span')
            if label=='no' and spans:
                errors.append(f'{nid}/{comp}: no but non-empty span')
            for s in spans:
                if not re.match(r'^\d+ \d+$', str(s)):
                    errors.append(f'{nid}/{comp}: bad span={s}')
    if errors:
        print(f'ERRORS ({len(errors)}):')
        for e in errors[:20]: print(f'  {e}')
        return False
    else:
        print(f'✓ Submission valid: {len(submission)} notes, all components correct.')
        return True


submission = build_submission(test_predictions, test_corpus.index.tolist())
validate_submission(submission)

# Stats
print('\nPrediction distribution:')
for comp in COMPONENTS:
    yes = sum(1 for v in submission.values() if v[comp]['label']=='yes')
    print(f'  {comp}: yes={yes} ({yes/len(submission):.1%})')

✓ Submission valid: 1959 notes, all components correct.

Prediction distribution:
  Definition 1: yes=263 (13.4%)
  Definition 2: yes=227 (11.6%)
  Rule B: yes=495 (25.3%)
  Rule C: yes=1228 (62.7%)


In [20]:
# ─── Save and Download Submission ─────────────────────────────────────────────
SUBMISSION_FILE = 'subtask_2_predictions_v2.json'

with open(SUBMISSION_FILE, 'w') as f:
    json.dump(submission, f, indent=2)

print(f'Saved: {SUBMISSION_FILE} ({os.path.getsize(SUBMISSION_FILE)//1024} KB)')

# Show sample
sample_id = list(submission.keys())[0]
print(f'\nSample ({sample_id}):')
print(json.dumps(submission[sample_id], indent=2))

#files.download(SUBMISSION_FILE)
#print('\nDownload started!')

Saved: subtask_2_predictions_v2.json (659 KB)

Sample (20):
{
  "Definition 1": {
    "label": "no",
    "span": []
  },
  "Definition 2": {
    "label": "yes",
    "span": [
      "4675 4684",
      "5917 5926",
      "4889 4896",
      "4689 4698",
      "5942 5964"
    ]
  },
  "Rule B": {
    "label": "yes",
    "span": [
      "367 384",
      "6591 6597"
    ]
  },
  "Rule C": {
    "label": "yes",
    "span": [
      "134 142",
      "217 226",
      "487 497",
      "617 626",
      "1374 1381",
      "2269 2275",
      "2294 2300"
    ]
  }
}


In [21]:
import json

# 1. Name of your current submission file (change if it's subtask_2.json)
input_file = "subtask_2_predictions_v2.json" 
output_file = "fixed_subtask_2_predictions_v2.json"

with open(input_file, "r") as f:
    predictions = json.load(f)

# 2. Loop through and truncate any spans greater than 20
for note_id, rules in predictions.items():
    for rule_name, rule_data in rules.items():
        if "span" in rule_data and len(rule_data["span"]) > 20:
            print(f"Fixing Note {note_id} | {rule_name}: Truncating {len(rule_data['span'])} spans down to 20.")
            rule_data["span"] = rule_data["span"][:20] # Keep only the first 20

# 3. Save the fixed predictions
with open(output_file, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Done! Please submit '{output_file}' to CodaBench.")

Fixing Note 7486 | Rule C: Truncating 24 spans down to 20.
Fixing Note 24284 | Rule C: Truncating 22 spans down to 20.
Done! Please submit 'fixed_subtask_2_predictions_v2.json' to CodaBench.


## Tuning Guide

### Understanding the v1 → v2 improvements

| Metric | v1 | Expected v2 |
|---|---|---|
| Label F1 | 0.654 | ~0.70+ |
| Span Exact F1 | 0.140 | ~0.35+ |
| Span Partial F1 | 0.363 | ~0.55+ |

The big wins come from:
1. **Rule B/C exact span matching** – finding the exact drug name token (e.g. `'Zolpidem'` at position `576–584`) instead of a long sentence.
2. **All occurrences** – returning every instance of the drug name in the note.

### If Definition 2 is still low
Add more indirect symptom keywords for a keyword-assisted baseline similar to Rule B/C:
```python
DEF2_KEYWORDS = ['restless', 'fatigue', 'fatigued', 'exhausted', 'lethargic', 
                 'tired', 'agitated', 'anxious', 'depressed', 'just wants to sleep']
```
But be careful — these words appear very commonly and will hurt precision.

### CodaBench submission
Upload `subtask_2_predictions_v2.json` at: https://www.codabench.org/competitions/15299